In [ ]:
# Practical 6: transfer learning - frozen VGG16 (ImageNet) + custom classifier on CIFAR-10
import tensorflow as tf
from keras import Sequential, layers
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.datasets import cifar10
from keras.optimizers import Adam

(X, y), (Xt, yt) = cifar10.load_data()
X, Xt = preprocess_input(X.astype('float32')), preprocess_input(Xt.astype('float32'))   # VGG's own preprocessing

base = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base.trainable = False                                                  # freeze all pre-trained layers
model = Sequential([base, layers.Flatten(), layers.Dense(256, activation='relu'), layers.Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=10, batch_size=32, validation_data=(Xt, yt))
print("Test accuracy:", model.evaluate(Xt, yt)[1])

In [ ]:
# (Optional, step e of the manual) fine-tune: unfreeze last conv block, very small learning rate
base.trainable = True
for l in base.layers[:-4]: l.trainable = False
model.compile(optimizer=Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=3, batch_size=32, validation_data=(Xt, yt))
print("Fine-tuned test accuracy:", model.evaluate(Xt, yt)[1])